# <font color="#418FDE" size="6.5" uppercase>**Capstone Automation**</font>

>Last update: 20260609.
    
By the end of this Lecture, you will be able to:
- Design a complete Python script for an applied chemical engineering task. 
- Integrate functions, data structures, loops, and file output into one workflow. 
- Evaluate project code for correctness, readability, and engineering usefulness. 


## **1. Project Design**

### **1.1. Problem Definition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_01_01.jpg?v=1780985995" width="250">



>* Define the script’s practical engineering purpose
>* Set clear boundaries and decision outcomes

>* Set a focused, manageable project scope
>* Define included inputs, logic, and exclusions

>* Tie automation to real engineering value
>* Define motivation, risks, and measurable usefulness



In [ ]:
#@title Python Code - Problem Definition

# Define the automation problem before coding.
# Connect the script to engineering usefulness.
# Keep project scope focused and testable.

# Store candidate capstone project definitions.
projects = [
    {
        "name": "Batch reactor run summary",
        "purpose": "Summarize conversion and temperature targets",

        "users": "process development engineers",
        "inputs": ["time", "temperature", "conversion"],
        "outputs": ["target flag", "summary report"],

        "included": ["performance indicators", "unusual run flags"],
        "excluded": ["kinetic model fitting", "scale-up design"],
        "automation_value": ["repeatability", "speed", "consistency"],
    },

    {
        "name": "Heat exchanger duty check",
        "purpose": "Compare measured and expected heat duty",
        "users": "plant support engineers",
        "inputs": ["flow rate", "temperatures", "heat capacity"],

        "outputs": ["duty comparison", "warning message"],
        "included": ["sensible heat balance", "percent deviation"],
        "excluded": ["fouling prediction", "mechanical design"],
        "automation_value": ["accuracy", "traceability", "consistency"],
    },
]

# Score each definition using simple checklist logic.
def score_definition(project):
    score = 0
    score += bool(project["purpose"])
    score += bool(project["users"])
    score += len(project["inputs"]) >= 2

    score += len(project["outputs"]) >= 1
    score += len(project["included"]) >= 1
    score += len(project["excluded"]) >= 1
    score += len(project["automation_value"]) >= 2

    return score

# Convert checklist scores into readable decisions.
def readiness_label(score, maximum):
    if score == maximum:
        return "well defined"

    if score >= maximum - 2:
        return "needs small edits"

    return "needs redesign"

# Evaluate the candidate project definitions.
maximum_score = 7
results = []

for project in projects:
    score = score_definition(project)
    label = readiness_label(score, maximum_score)

    results.append((project["name"], score, label))

# Print a compact capstone planning summary.
print("Capstone problem-definition checklist")

for name, score, label in results:
    print(f"{name}: {score}/{maximum_score}, {label}")

# Show one concise engineering problem statement.
best_project = max(projects, key=score_definition)
statement = (
    f"Chosen project: {best_project['name']} will help "
    f"{best_project['users']} {best_project['purpose'].lower()}."
)

print(statement)



### **1.2. Input Requirements**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_01_02.jpg?v=1780985993" width="250">



>* Identify all required engineering input data
>* Connect each input to project calculations

>* Specify input units, formats, ranges, and sources
>* Validate data to prevent engineering errors

>* Organized inputs make scripts easier to adapt
>* Clear guidance improves reliability for future users



In [ ]:
#@title Python Code - Input Requirements

# Project inputs control engineering calculation quality.
# Validation prevents misleading chemical process results.
# Organized requirements make scripts easier to extend.

# Define expected input requirements clearly.
required_inputs = {
    "flow_rate_L_min": {"unit": "L/min", "min": 0.1, "max": 500.0},
    "inlet_temp_C": {"unit": "deg C", "min": 0.0, "max": 150.0},
}

required_inputs.update({
    "outlet_temp_C": {"unit": "deg C", "min": 0.0, "max": 120.0},
    "density_kg_L": {"unit": "kg/L", "min": 0.5, "max": 1.5},
})

required_inputs.update({
    "heat_capacity_kJ_kg_C": {"unit": "kJ/kg C", "min": 1.0, "max": 6.0},
})

# Store one small design case.
design_case = {
    "flow_rate_L_min": 85.0,
    "inlet_temp_C": 92.0,
    "outlet_temp_C": 35.0,
}

# Add remaining physical properties.
design_case.update({
    "density_kg_L": 1.0,
    "heat_capacity_kJ_kg_C": 4.18,
})

# Validate each required input.
def validate_inputs(values, requirements):
    messages = []
    missing_keys = sorted(set(requirements) - set(values))

    for key in missing_keys:
        messages.append(f"Missing input: {key}")

    for key, rule in requirements.items():
        if key not in values:
            continue

        value = values[key]
        inside_range = rule["min"] <= value <= rule["max"]

        if not inside_range:
            messages.append(f"Range warning: {key}")

    return messages

# Calculate cooling duty after validation.
def cooling_duty_kW(values):
    flow_kg_min = values["flow_rate_L_min"] * values["density_kg_L"]
    delta_temp = values["inlet_temp_C"] - values["outlet_temp_C"]

    duty_kJ_min = flow_kg_min * values["heat_capacity_kJ_kg_C"] * delta_temp
    return duty_kJ_min / 60.0

# Check values before engineering calculations.
validation_messages = validate_inputs(design_case, required_inputs)
valid_case = len(validation_messages) == 0

# Print concise project input summary.
print("Input requirement check for cooling-duty project")
print(f"Required inputs: {len(required_inputs)}")
print(f"Provided inputs: {len(design_case)}")

# Report validation and calculation results.
if valid_case:
    duty = cooling_duty_kW(design_case)
    print("Validation result: all inputs acceptable")
    print(f"Estimated cooling duty: {duty:.1f} kW")
else:
    print("Validation result: review required")
    print("; ".join(validation_messages[:3]))



### **1.3. Output Planning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_01_03.jpg?v=1780985997" width="250">



>* Design outputs for engineering decisions
>* Highlight key values, units, and warnings

>* Tailor outputs to user needs
>* Create readable engineering records

>* Flag unreliable or unusual calculation results
>* Provide clear records for engineering decisions



In [ ]:
#@title Python Code - Output Planning

# Plan outputs before coding engineering automation.
# Use labels units and decisions.
# Save concise results for review.

import csv
from pathlib import Path

# Store design inputs in one dictionary.
case = {
    "case_name": "Wastewater polishing reactor",
    "flow_m3_per_day": 1200.0,
    "inlet_mg_per_L": 85.0,
    "target_mg_per_L": 20.0,
    "rate_constant_per_hr": 0.18,
}

# Calculate required removal and residence time.
removal_fraction = 1 - case["target_mg_per_L"] / case["inlet_mg_per_L"]
residence_time_hr = removal_fraction / case["rate_constant_per_hr"]
flow_m3_per_hr = case["flow_m3_per_day"] / 24.0

# Convert residence time into reactor volume.
reactor_volume_m3 = flow_m3_per_hr * residence_time_hr
actual_outlet_mg_per_L = case["inlet_mg_per_L"] * (1 - removal_fraction)
meets_target = actual_outlet_mg_per_L <= case["target_mg_per_L"]

# Build a readable engineering summary.
summary = {
    "case": case["case_name"],
    "flow_m3_per_day": case["flow_m3_per_day"],
    "required_removal_percent": 100 * removal_fraction,
    "residence_time_hr": residence_time_hr,
    "reactor_volume_m3": reactor_volume_m3,
    "outlet_mg_per_L": actual_outlet_mg_per_L,
    "meets_target": meets_target,
}

# Save structured output for later review.
output_path = Path("capstone_output_plan.csv")
with output_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=summary.keys())
    writer.writeheader()
    writer.writerow(summary)

# Print only the decision-focused results.
print("CAPSTONE OUTPUT PLAN")
print(f"Case: {summary['case']}")
print(f"Flow: {summary['flow_m3_per_day']:.0f} m3/day")
print(f"Removal needed: {summary['required_removal_percent']:.1f}%")
print(f"Residence time: {summary['residence_time_hr']:.2f} hr")
print(f"Reactor volume: {summary['reactor_volume_m3']:.1f} m3")
print(f"Outlet concentration: {summary['outlet_mg_per_L']:.1f} mg/L")
print(f"Meets discharge target: {summary['meets_target']}")
print(f"Saved file: {output_path.name}")



## **2. Integrated Workflow**

### **2.1. Function Workflow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_02_01.jpg?v=1780986004" width="250">



>* Make each function handle one clear task
>* Build readable workflows from defined function roles

>* Functions process structured data inside loops
>* New cases require data, not rewritten logic

>* Turn calculations into organized output files
>* Connect inputs, analysis, and deliverables



In [ ]:
#@title Python Code - Function Workflow

# Build one complete capstone workflow.
# Functions keep calculations readable.
# Results are saved for review.

import csv
from pathlib import Path

# Store small heat exchanger cases.
cases = [
    {"case": "A", "flow_kg_s": 1.2, "cp_kj_kgK": 4.18, "tin_C": 25, "tout_C": 60},
    {"case": "B", "flow_kg_s": 0.9, "cp_kj_kgK": 4.18, "tin_C": 30, "tout_C": 52},
    {"case": "C", "flow_kg_s": 1.5, "cp_kj_kgK": 3.90, "tin_C": 22, "tout_C": 70},
]

# Define the required heat duty.
def heat_duty_kw(stream):
    delta_T = stream["tout_C"] - stream["tin_C"]
    return stream["flow_kg_s"] * stream["cp_kj_kgK"] * delta_T

# Classify each engineering case.
def classify_case(duty_kw, target_kw):
    if duty_kw >= target_kw:
        return "meets target"
    return "below target"

# Format one row for saving.
def build_result_row(stream, target_kw):
    duty_kw = heat_duty_kw(stream)
    status = classify_case(duty_kw, target_kw)
    return {"case": stream["case"], "duty_kW": round(duty_kw, 1), "status": status}

# Loop through cases with functions.
target_kw = 150.0
results = []

# Build a result table list.
for stream in cases:
    result_row = build_result_row(stream, target_kw)
    results.append(result_row)

# Validate expected workflow size.
if len(results) != len(cases):
    raise ValueError("Every case must produce one result.")

# Save a compact CSV report.
output_path = Path("heat_exchanger_summary.csv")
fieldnames = ["case", "duty_kW", "status"]

# Write rows using built-in csv.
with output_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

# Print a short engineering summary.
print("Function workflow completed for", len(results), "cases.")
print("Target heat duty:", target_kw, "kW")

# Show only compact final results.
for row in results:
    print(row["case"], row["duty_kW"], "kW", row["status"])

# Confirm the saved deliverable.
print("Saved report:", output_path.name)



### **2.2. Loop Logic**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_02_02.jpg?v=1780986008" width="250">



>* Loops automate repeated engineering calculations
>* Each pass processes and updates results

>* Choose data structures that fit the task
>* Loop consistently through engineering data

>* Give each loop one clear purpose
>* Separate tasks for safer future changes



In [ ]:
#@title Python Code - Loop Logic

# Loop logic coordinates repeated engineering calculations.
# Functions keep each calculation easy to test.
# Results are stored before file output.

import csv
from pathlib import Path

# Define small candidate exchanger cases.
designs = [
    {"name": "HX-1", "area_m2": 24.0, "u_w_m2k": 420.0, "dt_k": 18.0},
    {"name": "HX-2", "area_m2": 30.0, "u_w_m2k": 390.0, "dt_k": 20.0},
    {"name": "HX-3", "area_m2": 18.0, "u_w_m2k": 510.0, "dt_k": 16.0},
]

# Calculate heat duty in kilowatts.
def heat_duty_kw(area_m2, u_w_m2k, dt_k):
    duty_watts = area_m2 * u_w_m2k * dt_k
    return duty_watts / 1000.0

# Classify each design using one rule.
def classify_design(duty_kw, target_kw):
    if duty_kw >= target_kw:
        return "meets target"
    return "below target"

# Set an engineering target safely.
target_kw = 200.0
results = []

# Loop through designs and store records.
for design in designs:
    duty_kw = heat_duty_kw(
        design["area_m2"], design["u_w_m2k"], design["dt_k"]
    )

    status = classify_design(duty_kw, target_kw)
    result = {"name": design["name"], "duty_kw": round(duty_kw, 1), "status": status}
    results.append(result)

# Write compact results to a CSV file.
output_path = Path("heat_exchanger_loop_results.csv")
with output_path.open("w", newline="") as file_handle:
    fieldnames = ["name", "duty_kw", "status"]
    writer = csv.DictWriter(file_handle, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(results)

# Summarize the loop outcome clearly.
passing_count = sum(1 for row in results if row["status"] == "meets target")
best_design = max(results, key=lambda row: row["duty_kw"])

# Print fewer than fifteen teaching lines.
print("Integrated loop processed", len(results), "exchanger designs.")
print("Designs meeting target:", passing_count)
print("Best design:", best_design["name"], best_design["duty_kw"], "kW")
print("CSV file written:", output_path.name)



### **2.3. Function Workflow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_02_03.jpg?v=1780986006" width="250">



>* Divide engineering tasks into clear functions
>* Connect functions to support sound analysis

>* Structured data carries information between functions
>* Reusable functions keep complex analyses clear

>* Reusable functions support loops and changing inputs
>* Saved outputs make results traceable and useful



In [ ]:
#@title Python Code - Function Workflow

# Build a complete function workflow example.
# Analyze small reactor trial data safely.
# Save concise engineering results to file.

import csv
from pathlib import Path

# Store tiny process data as dictionaries.
trials = [
    {"trial": "A", "feed_mol_h": 100.0, "product_mol_h": 78.0},
    {"trial": "B", "feed_mol_h": 120.0, "product_mol_h": 91.0},
    {"trial": "C", "feed_mol_h": 90.0, "product_mol_h": 70.0},
]

# Validate each trial before calculations.
def validate_trial(trial):
    required = ["trial", "feed_mol_h", "product_mol_h"]
    missing = [key for key in required if key not in trial]

    if missing:
        raise ValueError("Missing required trial fields")
    feed = float(trial["feed_mol_h"])
    product = float(trial["product_mol_h"])

    if feed <= 0 or product < 0:
        raise ValueError("Flow rates must be physical")
    return feed, product

# Calculate conversion for one reactor trial.
def calculate_conversion(trial):
    feed, product = validate_trial(trial)
    converted = feed - product

    conversion = converted / feed
    return {
        "trial": trial["trial"],
        "feed_mol_h": feed,
        "product_mol_h": product,
        "conversion_percent": 100.0 * conversion,
    }

# Summarize results for engineering review.
def summarize_results(results):
    conversions = [row["conversion_percent"] for row in results]
    average_conversion = sum(conversions) / len(conversions)

    best_result = max(results, key=lambda row: row["conversion_percent"])
    return average_conversion, best_result

# Write compact results to a CSV file.
def write_results(results, output_path):
    fieldnames = list(results[0].keys())

    with open(output_path, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

# Loop through data using the calculation function.
results = []
for trial in trials:
    results.append(calculate_conversion(trial))

# Create summary values from structured results.
average_conversion, best_result = summarize_results(results)
output_path = Path("reactor_conversion_summary.csv")
write_results(results, output_path)

# Print concise evidence of the workflow output.
print("Function workflow completed for", len(results), "trials.")
print("Average conversion:", round(average_conversion, 1), "%")
print("Best trial:", best_result["trial"], round(best_result["conversion_percent"], 1), "%")
print("Saved file:", output_path.name)



## **3. Code Review**

### **3.1. Testing Correctness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_03_01.jpg?v=1780986002" width="250">



>* Check results against engineering meaning
>* Compare outputs with trusted known answers

>* Test normal, boundary, and unusual cases
>* Handle bad inputs to prevent costly errors

>* Trace data flow and engineering logic
>* Check intermediate results and reproducibility



In [ ]:
#@title Python Code - Testing Correctness

# Review tests protect engineering automation decisions.
# Small examples reveal hidden calculation mistakes.
# Expected answers make correctness measurable.

import csv
import math
from pathlib import Path

# Define a heat duty calculation function.
def heat_duty_kw(flow_kg_s, cp_kj_kg_k, delta_t_k):
    if flow_kg_s < 0 or cp_kj_kg_k <= 0:
        raise ValueError("flow must be nonnegative and cp positive")

    return flow_kg_s * cp_kj_kg_k * delta_t_k

# Store clear test cases with expected results.
test_cases = [
    {"name": "normal metric", "flow": 2.0, "cp": 4.0, "dt": 10.0, "expected": 80.0},
    {"name": "zero temperature change", "flow": 3.0, "cp": 4.2, "dt": 0.0, "expected": 0.0},
    {"name": "zero flow boundary", "flow": 0.0, "cp": 4.2, "dt": 25.0, "expected": 0.0},
]

# Check each case against the trusted answer.
results = []
for case in test_cases:
    calculated = heat_duty_kw(case["flow"], case["cp"], case["dt"])
    passed = math.isclose(calculated, case["expected"], rel_tol=1e-9)

    results.append({"case": case["name"], "calculated_kw": calculated, "passed": passed})

# Check that invalid input fails clearly.
try:
    heat_duty_kw(-1.0, 4.2, 10.0)
    bad_input_passed = False
except ValueError:
    bad_input_passed = True

# Write a small review report file.
report_path = Path("correctness_review_report.csv")
with report_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["case", "calculated_kw", "passed"])
    writer.writeheader()

    writer.writerows(results)

# Summarize correctness without printing excessive detail.
total_passed = sum(item["passed"] for item in results)
all_passed = total_passed == len(results) and bad_input_passed

print("Correctness tests passed:", total_passed, "of", len(results))
print("Bad input handled clearly:", bad_input_passed)
print("Report file written:", report_path.name)
print("Overall engineering review status:", "PASS" if all_passed else "CHECK")



### **3.2. Readability Review**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_03_02.jpg?v=1780986000" width="250">



>* Readable code clearly communicates engineering purpose.
>* Organize workflow, names, data, and outputs.

>* Use comments to explain intent and assumptions
>* Choose meaningful names without cluttering code

>* Consistent formatting makes workflows easier to audit
>* Clear assumptions support reproducible engineering decisions



In [ ]:
#@title Python Code - Readability Review

# Review readability using a tiny capstone workflow.
# Chemical calculations need clear traceable structure.
# This example scores code review observations.

# Import standard tools for tabular review.
import csv
from pathlib import Path

# Store review criteria with engineering meaning.
review_items = [
    {"criterion": "clear_flow", "score": 4, "note": "inputs_to_outputs"},
    {"criterion": "meaningful_names", "score": 5, "note": "engineering_terms"},

    {"criterion": "useful_comments", "score": 3, "note": "assumptions_explained"},
    {"criterion": "consistent_format", "score": 4, "note": "spacing_matches"},
    {"criterion": "labeled_outputs", "score": 5, "note": "units_included"},
]

# Define a small reusable scoring function.
def summarize_readability(items, passing_score):
    scores = [item["score"] for item in items]
    if len(scores) == 0:
        return 0.0, "no_items"

    average_score = sum(scores) / len(scores)
    review_status = "passes" if average_score >= passing_score else "revise"
    return average_score, review_status

# Define safe project review settings.
maximum_score = 5
passing_score = 4.0
output_path = Path("readability_review.csv")

# Check that scores match expected review limits.
valid_scores = all(0 <= item["score"] <= maximum_score for item in review_items)
if not valid_scores:
    raise ValueError("Review scores must be between zero and five.")

# Calculate the readability summary.
average_score, review_status = summarize_readability(review_items, passing_score)
lowest_item = min(review_items, key=lambda item: item["score"])

# Write a compact review file.
with output_path.open("w", newline="") as file_object:
    fieldnames = ["criterion", "score", "note"]
    writer = csv.DictWriter(file_object, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(review_items)

# Print a concise review report.
print("Readability review for capstone automation")
print(f"Criteria reviewed: {len(review_items)}")
print(f"Average score: {average_score:.1f} of {maximum_score}")
print(f"Review status: {review_status}")
print(f"First improvement target: {lowest_item['criterion']}")
print(f"Review file written: {output_path.name}")



### **3.3. Engineering Usefulness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_05/Lecture_B/image_03_03.jpg?v=1780985999" width="250">



>* Code should support real engineering decisions
>* Inputs and outputs must fit practice

>* Handle messy data and unusual cases
>* Show assumptions, warnings, and engineering limits

>* Make code easy to adapt and understand
>* Support reliable decisions beyond one dataset



In [ ]:
#@title Python Code - Engineering Usefulness

# Review code for engineering usefulness.
# Connect calculations to plant decisions.
# Keep outputs concise and actionable.

import csv
from pathlib import Path

# Store small plant records as dictionaries.
records = [
    {"day": "Mon", "unit": "Reactor", "flow_m3": 120.0, "cod_in": 410.0, "cod_out": 88.0},
    {"day": "Tue", "unit": "Reactor", "flow_m3": 118.0, "cod_in": 405.0, "cod_out": 91.0},
    {"day": "Wed", "unit": "Reactor", "flow_m3": None, "cod_in": 430.0, "cod_out": 120.0},
    {"day": "Thu", "unit": "Clarifier", "flow_m3": 130.0, "cod_in": 390.0, "cod_out": 80.0},
    {"day": "Fri", "unit": "Clarifier", "flow_m3": 127.0, "cod_in": 395.0, "cod_out": 160.0},
]

# Define engineering limits and assumptions.
minimum_removal_percent = 70.0
required_fields = ["day", "unit", "flow_m3", "cod_in", "cod_out"]

# Check each record before calculations.
def validate_record(record, required_fields):
    missing = [field for field in required_fields if record.get(field) is None]
    numeric_ok = record.get("cod_in", 0) > 0 and record.get("cod_out", 0) >= 0
    return missing, numeric_ok

# Calculate removal efficiency safely.
def removal_percent(record):
    inlet = record["cod_in"]
    outlet = record["cod_out"]
    return 100.0 * (inlet - outlet) / inlet

# Build a concise engineering report.
summary_rows = []
warnings = []

# Review records for usefulness, not just correctness.
for record in records:
    missing, numeric_ok = validate_record(record, required_fields)
    if missing or not numeric_ok:
        warnings.append(f"{record['day']}: check missing or invalid data")
        continue
    removal = removal_percent(record)
    status = "OK" if removal >= minimum_removal_percent else "INVESTIGATE"
    summary_rows.append([record["day"], record["unit"], round(removal, 1), status])

# Save a shareable supervisor summary file.
output_path = Path("wastewater_usefulness_review.csv")
with output_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["day", "unit", "removal_percent", "engineering_status"])
    writer.writerows(summary_rows)

# Print only decision-focused review results.
print("Engineering usefulness review")
print(f"Usable records: {len(summary_rows)} of {len(records)}")
print(f"Warnings requiring follow-up: {len(warnings)}")
print(f"Days below target: {sum(row[3] == 'INVESTIGATE' for row in summary_rows)}")
print(f"Shareable output file: {output_path.name}")
print("Review note: outputs support follow-up decisions, not only arithmetic.")



# <font color="#418FDE" size="6.5" uppercase>**Capstone Automation**</font>


In this lecture, you learned to:
- Design a complete Python script for an applied chemical engineering task. 
- Integrate functions, data structures, loops, and file output into one workflow. 
- Evaluate project code for correctness, readability, and engineering usefulness. 

<font color='yellow'>Congratulations on completing this course!</font>